# Probe: kualitas Bahasa Indonesia + ketersediaan model NIM (alat keputusan)

Memutuskan generator Module 06 secara **empiris**. Fokus: **Nemotron 3 Nano** — model NVIDIA **paling baru & native** (MoE Hybrid Mamba-Attention, dirilis 15 Des 2025), dibandingkan dengan opsi berbasis Llama.

| Model | Lineage | Catatan |
|-------|---------|---------|
| `NVIDIA-Nemotron-3-Nano-30B-A3B` | **NVIDIA-native TERBARU** (MoE 30B/3B aktif) | id NIM dicari otomatis; dukungan ID belum dipastikan |
| `llama-3.3-nemotron-super-49b-v1.5` | Llama-based, NVIDIA-tuned | multilingual Llama 3.3 (ID kuat) |
| `llama-3.3-70b-instruct` | Meta | referensi ID terbaik (M05) |

> Sel akan **mencari id NIM** Nemotron 3 Nano otomatis (coba beberapa kandidat), lalu menampilkan jawaban tiap model untuk 3 prompt Bahasa Indonesia.

In [ ]:
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

In [ ]:
import re
from nvidia_utils import nim_client
client = nim_client()

def find_id(candidates):
    """Kembalikan id pertama yang VALID di NIM free tier (probe panggilan kecil)."""
    for mid in candidates:
        try:
            client.chat.completions.create(model=mid,
                messages=[{"role": "user", "content": "Halo"}], max_tokens=5)
            return mid
        except Exception:
            continue
    return None

# id NIM Nemotron 3 Nano belum pasti -> coba beberapa kandidat
NEMO3 = find_id([
    "nvidia/nemotron-3-nano-30b-a3b",
    "nvidia/nvidia-nemotron-3-nano-30b-a3b",
    "nvidia/nemotron-3-nano-30b-a3b-bf16",
    "nvidia/nvidia-nemotron-3-nano-30b-a3b-bf16",
    "nvidia/nemotron-3-nano-4b",
])
print("Nemotron 3 Nano di NIM:", NEMO3 or "TIDAK ditemukan di free tier (mungkin belum di-host)")

MODELS = {}
if NEMO3:
    MODELS["Nemotron-3-Nano (NVIDIA-native, TERBARU)"] = NEMO3
MODELS["Llama-Nemotron-Super-49B (NVIDIA-tuned)"] = "nvidia/llama-3.3-nemotron-super-49b-v1.5"
MODELS["Llama-3.3-70B (Meta)"] = "meta/llama-3.3-70b-instruct"
REASONING = {m for m in MODELS.values() if "nemotron" in m}   # model reasoning -> /no_think + strip

def ask(model, prompt):
    msgs = []
    if model in REASONING:
        msgs.append({"role": "system", "content": "/no_think"})
    msgs.append({"role": "user", "content": prompt})
    out = client.chat.completions.create(model=model, messages=msgs, temperature=0).choices[0].message.content
    return re.sub(r"<think>.*?</think>", "", out, flags=re.DOTALL).strip()

In [ ]:
PROMPTS = [
    ("Formal/edukasi", "Jelaskan apa itu retrieval-augmented generation (RAG) kepada lulusan SMA dalam 2 kalimat."),
    ("Layanan pelanggan", "Pelanggan bertanya: 'Kok pesanan saya belum sampai ya, padahal sudah 3 hari?' Jawab dengan sopan dan membantu."),
    ("Santai/idiomatik", "Beri satu tips singkat memilih GPU untuk belajar AI, pakai bahasa santai khas anak muda Indonesia."),
]
for judul, p in PROMPTS:
    print("=" * 90)
    print(f"PROMPT [{judul}]: {p}")
    print("=" * 90)
    for nama, model in MODELS.items():
        try:
            jawab = ask(model, p)
        except Exception as e:
            jawab = f"(error: {str(e)[:140]})"
        print(f"\n--- {nama} ---\n{jawab}\n")
    print()

## Keputusan

1. Apakah **Nemotron 3 Nano** muncul (id ditemukan di free tier)? Jika tidak → belum di-host, kita pertimbangkan Nemotron-Super atau Llama.
2. Jika muncul: apakah Bahasa Indonesia-nya **natural & benar**? 
   - **Bagus** → pakai Nemotron 3 Nano untuk Module 06 (paling NVIDIA-native + terbaru).
   - **Kaku/keliru** → pakai opsi yang ID-nya lebih kuat.

Beri tahu hasilnya — aku set model Module 06 sesuai bukti.